# Room Occupancy Classification — Model Training

This notebook downloads the original Room Occupancy Estimation dataset from the UCI Machine Learning Repository, validates and splits it by complete day, trains five classifiers, evaluates them on the same held-out day, and saves the fitted models and test data used by the Streamlit app.

## 1. Imports and configuration

The timestamp columns are used only to keep whole days together in the split; the models receive the 16 numeric sensor readings.

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Search upward from the current folder for requirements.txt to locate the project root.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.txt").is_file()),
    Path.cwd(),
)
# Folder where the fitted models are saved for the Streamlit app.
MODEL_DIR = ROOT / "model"
# Direct link to the original UCI Room Occupancy Estimation zip archive.
DATA_URL = "https://archive.ics.uci.edu/static/public/864/room+occupancy+estimation.zip"
# Name of the target column: the number of occupants in the room (0 to 3).
TARGET = "Room_Occupancy_Count"
# The 16 numeric sensor columns used as model inputs. Date and Time are not included.
FEATURES = [
    "S1_Temp", "S2_Temp", "S3_Temp", "S4_Temp",
    "S1_Light", "S2_Light", "S3_Light", "S4_Light",
    "S1_Sound", "S2_Sound", "S3_Sound", "S4_Sound",
    "S5_CO2", "S5_CO2_Slope", "S6_PIR", "S7_PIR",
]

print(f"Project directory: {ROOT}")

Project directory: c:\Users\kns317\OneDrive - Sky\Documents\BITS\ML\assignment2


## 2. Load the original UCI dataset

The source archive is downloaded directly from UCI. The notebook does not substitute a processed or randomly sampled dataset.

In [2]:
# Download the archive with a timeout and a User-Agent header so the request is not rejected.
request = urllib.request.Request(DATA_URL, headers={"User-Agent": "ML-assignment/1.0"})
with urllib.request.urlopen(request, timeout=60) as response:
    archive_bytes = response.read()

# Open the downloaded zip in memory and read the first CSV file found inside it.
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    csv_files = [name for name in archive.namelist() if name.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("The UCI archive does not contain a CSV dataset.")
    with archive.open(csv_files[0]) as csv_file:
        data = pd.read_csv(csv_file)

print(f"Dataset shape: {data.shape[0]:,} rows × {data.shape[1]} columns")
display(data.head())

Dataset shape: 10,129 rows × 19 columns


,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
0,2017/12/22,10:49:41,24.94,24.75,24.56,25.38,121,34,53,40,0.08,0.19,0.06,0.06,390,0.769231,0,0,1
1,2017/12/22,10:50:12,24.94,24.75,24.56,25.44,121,33,53,40,0.93,0.05,0.06,0.06,390,0.646154,0,0,1
2,2017/12/22,10:50:42,25.00,24.75,24.50,25.44,121,34,53,40,0.43,0.11,0.08,0.06,390,0.519231,0,0,1
3,2017/12/22,10:51:13,25.00,24.75,24.56,25.44,121,34,53,40,0.41,0.10,0.10,0.09,390,0.388462,0,0,1
4,2017/12/22,10:51:44,25.00,24.75,24.56,25.44,121,34,54,40,0.18,0.06,0.06,0.06,390,0.253846,0,0,1


## 3. Validate data and create a whole-day holdout

A random row split would put neighboring 30-second readings in both partitions. Instead, the first date that contains every class and leaves every class in the remaining dates is reserved for testing.

In [3]:
# Make sure the dataset has every column the notebook uses before going further.
required_columns = set(FEATURES + [TARGET, "Date", "Time"])
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"UCI dataset is missing expected columns: {sorted(missing_columns)}")

# Sort chronologically so the day-based split and any time-ordered checks behave predictably.
data = data.sort_values(["Date", "Time"]).reset_index(drop=True)
# Convert sensor columns to numbers. Non-numeric entries become NaN so the check below can catch them.
for feature in FEATURES:
    data[feature] = pd.to_numeric(data[feature], errors="coerce")
data[TARGET] = pd.to_numeric(data[TARGET], errors="coerce")
if data[FEATURES + [TARGET]].isna().any().any():
    raise ValueError("Sensor features and target must be numeric and contain no missing values.")
if not data[TARGET].mod(1).eq(0).all():
    raise ValueError("Occupancy target values must be whole numbers.")
# Use integer class labels (0 to 3) from here on.
data[TARGET] = data[TARGET].astype(int)

# All occupancy classes found in the data; used to check that the holdout day is valid.
class_labels = sorted(data[TARGET].unique().tolist())
all_classes = set(class_labels)
# Pick the first date that contains every class, and where the other dates still contain every class.
test_day = None
for day, day_rows in data.groupby("Date", sort=True):
    remaining_classes = set(data.loc[data["Date"] != day, TARGET].unique())
    if set(day_rows[TARGET].unique()) == all_classes and remaining_classes == all_classes:
        test_day = str(day)
        break
if test_day is None:
    raise ValueError("Could not find a full-day holdout containing every target class.")

# Split by whole day: rows from the chosen date form the test set, all other rows train the models.
is_test = data["Date"].eq(test_day)
train_rows = data.loc[~is_test]
test_rows = data.loc[is_test]
x_train, y_train = train_rows[FEATURES], train_rows[TARGET]
x_test, y_test = test_rows[FEATURES], test_rows[TARGET]

print(f"Training rows: {len(train_rows):,}")
print(f"Held-out date: {test_day} ({len(test_rows):,} rows)")
print(f"Classes: {class_labels}")
print(f"Predictors: {len(FEATURES)} sensor readings (Date and Time excluded)")
display(pd.DataFrame({"Train": y_train.value_counts().sort_index(), "Test": y_test.value_counts().sort_index()}))

Training rows: 8,667
Held-out date: 2017/12/22 (1,462 rows)
Classes: [0, 1, 2, 3]
Predictors: 16 sensor readings (Date and Time excluded)


,Train,Test
Room_Occupancy_Count,,
0,7591,637
1,254,205
2,491,257
3,331,363


## 4. Define and train the five classifiers

Scaling is included inside pipelines for the scale-sensitive classifiers. This ensures preprocessing is fitted only on training rows and is applied consistently when the app loads each saved model.

In [ ]:
# Each entry maps a model key (used as the saved file name) to (display name, estimator).
# Scale-sensitive models (logistic regression, KNN, Naive Bayes, SVM) are wrapped with StandardScaler in a pipeline.
model_specs = {
    "logistic_regression": (
        "Logistic Regression",
        Pipeline([
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
        ]),
    ),
    "decision_tree": (
        "Decision Tree",
        DecisionTreeClassifier(random_state=42),
    ),
    "knn": (
        "K-Nearest Neighbors",
        Pipeline([
            ("scale", StandardScaler()),
            ("classifier", KNeighborsClassifier(n_neighbors=7, weights="distance")),
        ]),
    ),
    "gaussian_nb": (
        "Gaussian Naive Bayes",
        Pipeline([
            ("scale", StandardScaler()),
            ("classifier", GaussianNB()),
        ]),
    ),
    "support_vector_machine": (
        "Support Vector Machine",
        Pipeline([
            ("scale", StandardScaler()),
            ("classifier", CalibratedClassifierCV(
                SVC(kernel="rbf", random_state=42), cv=3, ensemble=False
            )),
        ]),
    ),
}

# Create the model folder and save the held-out day as test_data.csv for the app to evaluate.
MODEL_DIR.mkdir(exist_ok=True)
# Use LF line endings here; text-mode writing on Windows would otherwise add a second carriage return.
(ROOT / "test_data.csv").write_text(
    test_rows[FEATURES + [TARGET]].to_csv(index=False, lineterminator="\n"),
    encoding="utf-8",
)

# Fit each model on the training rows, score it on the held-out day, and save it to disk.
metric_rows = {}
for model_key, (model_name, estimator) in model_specs.items():
    estimator.fit(x_train, y_train)
    predictions = estimator.predict(x_test)
    probabilities = estimator.predict_proba(x_test)
    metric_rows[model_key] = {
        "Accuracy": accuracy_score(y_test, predictions),
        "AUC": roc_auc_score(
            y_test,
            probabilities,
            labels=class_labels,
            multi_class="ovr",
            average="macro",
        ),
        "Precision": precision_score(y_test, predictions, average="macro", zero_division=0),
        "Recall": recall_score(y_test, predictions, average="macro", zero_division=0),
        "F1": f1_score(y_test, predictions, average="macro", zero_division=0),
        "MCC": matthews_corrcoef(y_test, predictions),
    }
    joblib.dump(estimator, MODEL_DIR / f"{model_key}.joblib")
    print(f"Saved {model_name} ({model_key})")

print(f"\nModels and test data written to {MODEL_DIR}")

Saved Logistic Regression (logistic_regression)
Saved Decision Tree (decision_tree)
Saved K-Nearest Neighbors (knn)
Saved Gaussian Naive Bayes (gaussian_nb)
Saved Support Vector Machine (support_vector_machine)

Artifacts written to c:\Users\kns317\OneDrive - Sky\Documents\BITS\ML\assignment2\model


## 5. Compare model performance

Precision, recall, and F1 use macro averaging so each occupancy class contributes equally. AUC is macro one-vs-rest.

In [5]:
# Build the comparison table with one row per model and format the metric values to four decimals.
comparison = pd.DataFrame.from_dict(metric_rows, orient="index")
comparison.insert(0, "Model", [model_specs[key][0] for key in comparison.index])
display(comparison.style.format({column: "{:.4f}" for column in comparison.columns if column != "Model"}))

# Identify the best model by macro F1 and by macro one-vs-rest AUC.
winner = comparison["F1"].idxmax()
auc_winner = comparison["AUC"].idxmax()
print(f"Highest macro F1: {model_specs[winner][0]} ({comparison.loc[winner, 'F1']:.4f})")
print(f"Highest macro one-vs-rest AUC: {model_specs[auc_winner][0]} ({comparison.loc[auc_winner, 'AUC']:.4f})")

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
logistic_regression,Logistic Regression,0.8953,0.9837,0.9037,0.8937,0.8771,0.8635
decision_tree,Decision Tree,0.8181,0.8742,0.8344,0.8042,0.7798,0.7657
knn,K-Nearest Neighbors,0.9083,0.9883,0.9035,0.9007,0.8922,0.8751
gaussian_nb,Gaussian Naive Bayes,0.6306,0.7938,0.5942,0.6656,0.5897,0.5462
support_vector_machine,Support Vector Machine,0.8208,0.9937,0.8735,0.8191,0.7870,0.7798


Highest macro F1: K-Nearest Neighbors (0.8922)
Highest macro one-vs-rest AUC: Support Vector Machine (0.9937)


## Generated artifacts

`model/` contains the five fitted estimators and `test_data.csv` contains the held-out labeled day. The Streamlit app loads the saved estimators and uses the feature, target, class, and model-name configuration in `app.py`; it does not retrain the models when evaluating data.